# 第6章：Reward Modeling

## 本章目标
- 理解偏好数据 (preference data) 的格式和收集方式
- 理解 Bradley-Terry 模型（Reward Model 的理论基础）
- 使用 trl 训练一个 Reward Model

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install torch transformers trl peft datasets accelerate bitsandbytes
else:
    print("本地环境运行，请确保已按 intro.md 配置好环境")

## Bradley-Terry 模型速览

Reward Model 的理论基础：给定两个回答 (chosen y_w, rejected y_l)，我们希望模型对 chosen 打分高于 rejected。

Bradley-Terry 模型：
P(y_w > y_l) = σ(r(y_w) - r(y_l))

其中 r(·) 是 Reward Model 的输出分数，σ 是 sigmoid 函数。

训练 loss = -log P(y_w > y_l)，让 chosen 的分数尽可能高于 rejected。

参考：[InstructGPT](https://arxiv.org/abs/2203.02155)

In [ ]:
from datasets import load_dataset

dataset = load_dataset("Anthropic/hh-rlhf", split="train[:3000]")
print(f"数据集大小: {len(dataset)}")
print(f"字段: {list(dataset[0].keys())}")
print(f"\nChosen 示例:\n{dataset[0]['chosen'][:200]}...")
print(f"\nRejected 示例:\n{dataset[0]['rejected'][:200]}...")

In [ ]:
from transformers import AutoTokenizer

model_name = "Qwen/Qwen2.5-0.5B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
from transformers import AutoModelForSequenceClassification
from trl import RewardTrainer, RewardConfig

reward_model = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=1, torch_dtype=torch.float16, device_map="auto"
)

training_args = RewardConfig(
    output_dir="./reward_model_output",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    learning_rate=1e-5,
    logging_steps=10,
    save_strategy="no",
    report_to="none",
    max_length=512,
)

trainer = RewardTrainer(
    model=reward_model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
)
trainer.train()
print("Reward Model 训练完成")

In [ ]:
import torch

def score_response(reward_model, tokenizer, text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512).to(reward_model.device)
    with torch.no_grad():
        score = reward_model(**inputs).logits[0, 0].item()
    return score

sample = dataset[0]
score_chosen = score_response(reward_model, tokenizer, sample["chosen"])
score_rejected = score_response(reward_model, tokenizer, sample["rejected"])
print(f"Chosen score:  {score_chosen:.4f}")
print(f"Rejected score: {score_rejected:.4f}")
print(f"Chosen > Rejected: {score_chosen > score_rejected}")

## 练习

1. 在更多数据上训练 Reward Model，观察 chosen > rejected 的准确率
2. 尝试用更大的 base model (如 Qwen2.5-1.5B) 训练 Reward Model
3. 分析 Reward Model 打分错误的 case，思考原因

## 延伸阅读

- [InstructGPT: Training language models to follow instructions](https://arxiv.org/abs/2203.02155)
- [Anthropic HH-RLHF 数据集](https://huggingface.co/datasets/Anthropic/hh-rlhf)
- [trl RewardTrainer 文档](https://huggingface.co/docs/trl/reward_trainer)